# Housing Prices Prediction

### Install Dependencies

We begin by installing the core libraries for our analysis:
- `pandas`: For data manipulation and analysis.
- `numpy`: For numerical operations.
- `seaborn` & `matplotlib`: For creating data visualizations.
- `scikit-learn`: For machine learning algorithms.
- `kagglehub`: For loading datasets from Kaggle.
- `statsmodels`: For statistical modeling.

In [ ]:
%pip install pandas matplotlib seaborn numpy scikit-learn kagglehub statsmodels

### Import Libraries

We import the necessary libraries for data manipulation, visualization, and machine learning. This includes `pandas` for data handling, `numpy` for numerical operations, `seaborn` and `matplotlib` for plotting, and various scikit-learn modules for modeling.

In [ ]:
import os
import math
import numpy as np
import pandas as pd
import seaborn as sns
from datetime import datetime
from IPython.display import display

from statsmodels.formula import api
from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = [10, 6]

import warnings
warnings.filterwarnings('ignore')

### Load Data

We use `kagglehub.load_dataset` to fetch the housing prices data from Kaggle. The data is loaded into a Pandas DataFrame (`df`), which is the standard structure for data analysis in Python. We also define the target variable as 'price' and identify the features.

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "Housing.csv"

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "yasserh/housing-prices-dataset",
    file_path,
)

print("First 5 records:", df.head())

display(df.head())

target = 'price'
features = [i for i in df.columns if i not in [target]]

original_df = df.copy(deep=True)

### Dataset Schema Analysis

The `df.info()` method provides a high-level overview of our DataFrame. It shows the number of entries (rows), the column names, and the data type of each column (e.g., `int64`, `object`, `float64`). It also reveals missing values via the 'Non-Null Count'.

In [ ]:
df.info()


### Unique Values per Column

`df.nunique()` returns the number of unique values in each column. This helps us distinguish between categorical (few unique values) and numerical features.

In [ ]:
df.nunique()

### Missing Data Detection

We use `df.isnull().sum()` to create a count of missing entries for every column. This helps us identify which columns need imputation or removal.

In [ ]:
df.isnull().sum()

In [ ]:
target = 'price'
features = [i for i in df.columns if i not in [target]]

original_df = df.copy(deep=True)

print('\n\033[1mInference:\033[0m The Datset consists of {} features & {} samples.'.format(df.shape[1], df.shape[0]))
display(df.head())

In [ ]:
nu = df[features].nunique().sort_values()
nf = []; cf = []; 

for i in range(df[features].shape[1]):
    if nu.values[i]<=16:cf.append(nu.index[i])
    else: nf.append(nu.index[i])

print('\n\033[1mInference:\033[0m The Datset has {} numerical & {} categorical features.'.format(len(nf),len(cf)))

### Statistical Summary

The `df.describe()` method calculates descriptive statistics for all numerical columns. It includes the mean, standard deviation, minimum, maximum, and the values at the 25%, 50% (median), and 75% percentiles.

In [ ]:
display(df.describe())

### Exploratory Data Analysis (EDA)

### Target Variable Distribution

We plot the distribution of the target variable 'price' using Seaborn's `distplot`. This helps us understand the spread, central tendency, and skewness of house prices.

In [ ]:
#EDA
plt.figure(figsize=[8,4])
sns.distplot(df[target], color='g',hist_kws=dict(edgecolor="black", linewidth=2), bins=30)
plt.title('Target Variable Distribution - Median Value of Homes ($1Ms)')
plt.show()

### Categorical Features Visualization

We use Seaborn's `countplot` to visualize the distribution of categorical features. This shows the frequency of each category in the dataset.

In [ ]:
#Plot the categorical features
n = 3
rows = math.ceil(len(cf) / n)
plt.figure(figsize=(15, 3 * rows))

for i, col_name in enumerate(cf):
    plt.subplot(rows, n, i + 1)

    sns.countplot(data=df, x=col_name, hue=col_name, palette='Set2', legend=False)

    plt.title(f'Distribution of {col_name}')

plt.tight_layout()
plt.show()

### Numerical Features Visualization

We plot the distributions and box plots of numerical features using `histplot` (with KDE) and `boxplot` to check for normality, skewness, and potential outliers.

In [ ]:
#Plot the numerical features
print('\033[1mNumeric Features Distribution'.center(130))

n=3
rows = math.ceil(len(nf) / n)

# Distribution plots
plt.figure(figsize=(15, 3 * rows))
for i, col_name in enumerate(nf):
    plt.subplot(rows, n, i + 1)
    sns.histplot(df[col_name], kde=True, edgecolor="black", linewidth=2, bins=10, color=np.random.rand(3))
    plt.title(f'Distribution of {col_name}')
plt.tight_layout()
plt.show()

# Box plots
plt.figure(figsize=(15, 3 * rows))
for i, col_name in enumerate(nf):
    plt.subplot(rows, n, i + 1)
    sns.boxplot(y=df[col_name])
    plt.title(f'Box Plot of {col_name}')
plt.tight_layout()
plt.show()

### Pairwise Relationships

We use Seaborn's `pairplot` to visualize pairwise relationships between all features, with KDE contours on the upper triangle. This helps identify correlations, clusters, and patterns in the data.

In [ ]:
g = sns.pairplot(df)
plt.title('Pairplots for all the Feature')
g.map_upper(sns.kdeplot, levels=4, color=".2")
plt.show()

### Data Processing

### Duplicate Values Check

We check for duplicate rows using `drop_duplicates()`. If duplicates exist, they are removed to prevent biasing the analysis and model training.

In [ ]:
rs, cs = original_df.shape

df.drop_duplicates(inplace=True)

if df.shape == (rs, cs):
    print('\n\033[1mInference:\033[0m The dataset doesn\'t have any duplicates')
else:
    print(f'\n\033[1mInference:\033[0m Number of duplicates dropped/fixed ---> {rs - df.shape[0]}')

### Check for Empty Elements

We create a DataFrame summarizing null values per column, including percentages. This helps quantify the extent of missing data in the dataset.

In [ ]:
nvc = pd.DataFrame(df.isnull().sum().sort_values(), columns=['Total Null Values'])
nvc['Percentage'] = round(nvc['Total Null Values'] / df.shape[0], 3) * 100
print(nvc)

### Converting Categorical Columns to Numeric

We use `pd.get_dummies()` for one-hot encoding binary features and dummy encoding for multi-category features. This converts categorical data into numerical format suitable for machine learning algorithms.

In [ ]:
df3 = df.copy()

ecc = nvc[nvc['Percentage'] != 0].index.values
fcc = [i for i in cf if i not in ecc]

oh = True
dm = True
for i in fcc:
    if df3[i].nunique() == 2:
        if oh:
            print("\033[1mOne-Hot Encoding on features:\033[0m")
            oh = False
        print(i)
        dummy_col = pd.get_dummies(df3[i], drop_first=True, prefix=str(i)).astype(int)
        df3 = pd.concat([df3.drop([i], axis=1), dummy_col], axis=1)
    elif (df3[i].nunique() > 2 and df3[i].nunique() < 17):
        if dm:
            print("\n\033[1mDummy Encoding on features:\033[0m")
            dm = False
        print(i)
        df3 = pd.concat([df3.drop([i], axis=1), pd.DataFrame(pd.get_dummies(df3[i], drop_first=True, prefix=str(i))).astype(int)], axis=1)

df3.shape

In [ ]:
df3.head()

### Removal of Outliers

We remove outliers from numerical features using the IQR method. Values outside the range [Q1 - 1.5*IQR, Q3 + 1.5*IQR] are filtered out to reduce the impact of extreme values on the model.

In [ ]:
df1 = df3.copy()

features1 = nf

for i in features1:
    Q1 = df1[i].quantile(0.25)
    Q3 = df1[i].quantile(0.75)
    IQR = Q3 - Q1
    df1 = df1[df1[i] <= (Q3 + (1.5 * IQR))]
    df1 = df1[df1[i] >= (Q1 - (1.5 * IQR))]
    df1 = df1.reset_index(drop=True)

display(df1.head())
print('\n\033[1mInference:\033[0m\nBefore removal of outliers, The dataset had {} samples.'.format(df3.shape[0]))
print('After removal of outliers, The dataset now has {} samples.'.format(df1.shape[0]))

### Final Dataset Size After Preprocessing

We display a pie chart showing the proportion of retained vs dropped samples after preprocessing steps like outlier removal.

In [ ]:
#Final Dataset size after performing Preprocessing

df = df1.copy()
df.columns=[i.replace('-','_') for i in df.columns]

plt.title('Final Dataset')
plt.pie([df.shape[0], original_df.shape[0]-df.shape[0]], radius = 1, labels=['Retained','Dropped'], counterclock=False,
        autopct='%1.1f%%', pctdistance=0.9, explode=[0,0], shadow=True)
plt.pie([df.shape[0]], labels=['100%'], labeldistance=-0, radius=0.78)
plt.show()

print(f'\n\033[1mInference:\033[0m After the cleanup process, {original_df.shape[0]-df.shape[0]} samples were dropped, \
while retaining {round(100 - (df.shape[0]*100/(original_df.shape[0])),2)}% of the data.')

### Data Splitting

We split the dataset into training (80%) and testing (20%) sets using `train_test_split`. This allows us to train the model on one portion and evaluate its performance on unseen data.

In [ ]:
df = df1.copy()
df.columns = [i.replace('-', '_') for i in df.columns]

X = df.drop([target], axis=1)
Y = df[target]
Train_X, Test_X, Train_Y, Test_Y = train_test_split(X, Y, train_size=0.8, test_size=0.2, random_state=100)
Train_X.reset_index(drop=True, inplace=True)

print('Original set  ---> ', X.shape, Y.shape, '\nTraining set  ---> ', Train_X.shape, Train_Y.shape, '\nTesting set   ---> ', Test_X.shape, '', Test_Y.shape)

### Feature Scaling

We apply standardization using `StandardScaler` to transform features to have a mean of 0 and standard deviation of 1. This is crucial for models like linear regression that are sensitive to feature scales.

In [ ]:
#Feature Scaling (Standardization)

std = StandardScaler()

print('\033[1mStandardardization on Training set'.center(120))
Train_X_std = std.fit_transform(Train_X)
Train_X_std = pd.DataFrame(Train_X_std, columns=X.columns)
display(Train_X_std.describe())

print('\n','\033[1mStandardardization on Testing set'.center(120))
Test_X_std = std.transform(Test_X)
Test_X_std = pd.DataFrame(Test_X_std, columns=X.columns)
display(Test_X_std.describe())

### Feature Selection/Extraction

### Correlation Matrix

We visualize the correlation matrix using a heatmap. This helps identify multicollinearity and relationships between features and the target variable.

In [ ]:
# Feature Selection/Extraction
#Checking the correlation

print('\033[1mCorrelation Matrix'.center(100))
plt.figure(figsize=[25,20])
sns.heatmap(df.corr(), annot=True, vmin=-1, vmax=1, center=0) #cmap='BuGn'
plt.show()

### Testing Linear Regression with Statsmodels

We use `statsmodels.api.ols` to perform ordinary least squares regression and obtain a detailed summary including coefficients, p-values, and goodness-of-fit statistics.

In [ ]:
Train_xy = pd.concat([Train_X_std, Train_Y.reset_index(drop=True)], axis=1)
a = Train_xy.columns.values

API = api.ols(formula='{} ~ {}'.format(target, ' + '.join(i for i in Train_X.columns)), data=Train_xy).fit()
print(API.summary())

### Manual Method - Variance Inflation Factor (VIF)

We calculate VIF for each feature and iteratively remove those with VIF > 1 to reduce multicollinearity, monitoring RMSE on train and test sets.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
Trr = []
Tss = []
n = 3
order = ['ord-' + str(i) for i in range(2, n)]

DROP = []

while True:
    X = Train_X_std.drop(DROP, axis=1)
    if X.shape[1] <= 1:
        break
    vif = pd.DataFrame()
    vif['Features'] = X.columns
    vif['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    vif['VIF'] = round(vif['VIF'], 2)
    vif = vif.sort_values(by="VIF", ascending=False)
    vif.reset_index(drop=True, inplace=True)
    if vif.loc[0, 'VIF'] <= 1:
        break
    DROP.append(vif.loc[0, 'Features'])
    LR = LinearRegression()
    LR.fit(Train_X_std.drop(DROP, axis=1), Train_Y)

    pred1 = LR.predict(Train_X_std.drop(DROP, axis=1))
    pred2 = LR.predict(Test_X_std.drop(DROP, axis=1))

    Trr.append(np.sqrt(mean_squared_error(Train_Y, pred1)))
    Tss.append(np.sqrt(mean_squared_error(Test_Y, pred2)))

print('Dropped Features --> ', DROP)
plt.plot(Trr, label='Train RMSE')
plt.plot(Tss, label='Test RMSE')
plt.legend()
plt.grid()
plt.show()

### Automatic Method - Recursive Feature Elimination (RFE)

We apply RFE to automatically select features by recursively training the model and removing the weakest features, evaluating RMSE across different numbers of features.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
Trr = []
Tss = []
n = 3
order = ['ord-' + str(i) for i in range(2, n)]
Trd = pd.DataFrame(np.zeros((10, n - 2)), columns=order)
Tsd = pd.DataFrame(np.zeros((10, n - 2)), columns=order)

m = df.shape[1] - 2
for i in range(m):
    lm = LinearRegression()
    rfe = RFE(lm, n_features_to_select=Train_X_std.shape[1] - i)
    rfe = rfe.fit(Train_X_std, Train_Y)

    LR = LinearRegression()
    LR.fit(Train_X_std.loc[:, rfe.support_], Train_Y)

    pred1 = LR.predict(Train_X_std.loc[:, rfe.support_])
    pred2 = LR.predict(Test_X_std.loc[:, rfe.support_])

    Trr.append(np.sqrt(mean_squared_error(Train_Y, pred1)))
    Tss.append(np.sqrt(mean_squared_error(Test_Y, pred2)))

plt.plot(Trr, label='Train RMSE')
plt.plot(Tss, label='Test RMSE')
plt.legend()
plt.grid()
plt.show()

### Feature Elimination using PCA Decomposition

We use PCA to transform features into principal components and assess how many components are needed to explain the variance, plotting cumulative explained variance.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA().fit(Train_X_std)

fig, ax = plt.subplots(figsize=(8, 6))
x_values = range(1, pca.n_components_ + 1)
ax.bar(x_values, pca.explained_variance_ratio_, lw=2, label='Explained Variance')
ax.plot(x_values, np.cumsum(pca.explained_variance_ratio_), lw=2, label='Cumulative Explained Variance', color='red')
plt.plot([0, pca.n_components_ + 1], [0.9, 0.9], 'g--')
ax.set_title('Explained variance of components')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Explained Variance')
plt.legend()
plt.grid()
plt.show()

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import PolynomialFeatures
Trr = []
Tss = []
n = 3
order = ['ord-' + str(i) for i in range(2, n)]
Trd = pd.DataFrame(np.zeros((10, n - 2)), columns=order)
Tsd = pd.DataFrame(np.zeros((10, n - 2)), columns=order)
m = df.shape[1] - 1

for i in range(m):
    pca = PCA(n_components=Train_X_std.shape[1] - i)
    Train_X_std_pca = pca.fit_transform(Train_X_std)
    Test_X_std_pca = pca.fit_transform(Test_X_std)

    LR = LinearRegression()
    LR.fit(Train_X_std_pca, Train_Y)

    pred1 = LR.predict(Train_X_std_pca)
    pred2 = LR.predict(Test_X_std_pca)

    Trr.append(round(np.sqrt(mean_squared_error(Train_Y, pred1)), 2))
    Tss.append(round(np.sqrt(mean_squared_error(Test_Y, pred2)), 2))

plt.plot(Trr, label='Train RMSE')
plt.plot(Tss, label='Test RMSE')
plt.legend()
plt.grid()
plt.show()

In [ ]:
lm = LinearRegression()
rfe = RFE(lm, n_features_to_select=Train_X_std.shape[1] - 5)
rfe = rfe.fit(Train_X_std, Train_Y)

LR = LinearRegression()
LR.fit(Train_X_std.loc[:, rfe.support_], Train_Y)

pred1 = LR.predict(Train_X_std.loc[:, rfe.support_])
pred2 = LR.predict(Test_X_std.loc[:, rfe.support_])

print(np.sqrt(mean_squared_error(Train_Y, pred1)))
print(np.sqrt(mean_squared_error(Test_Y, pred2)))

In [ ]:
Model_Evaluation_Comparison_Matrix = pd.DataFrame(np.zeros([5,8]), columns=['Train-R2','Test-R2','Train-RSS','Test-RSS',
                                                                            'Train-MSE','Test-MSE','Train-RMSE','Test-RMSE'])
rc=np.random.choice(Train_X_std.loc[:,Train_X_std.nunique()>=50].columns.values,1,replace=False)
def Evaluate(n, pred1,pred2):
    plt.figure(figsize=[15,6])
    for e,i in enumerate(rc):
        plt.subplot(2,3,e+1)
        plt.scatter(y=Train_Y, x=Train_X_std[i], label='Actual')
        plt.scatter(y=pred1, x=Train_X_std[i], label='Prediction')
        plt.legend()
    plt.show()

    print('\n\n{}Training Set Metrics{}'.format('-'*20, '-'*20))
    print('\nR2-Score on Training set --->',round(r2_score(Train_Y, pred1),20))
    print('Residual Sum of Squares (RSS) on Training set  --->',round(np.sum(np.square(Train_Y-pred1)),20))
    print('Mean Squared Error (MSE) on Training set       --->',round(mean_squared_error(Train_Y, pred1),20))
    print('Root Mean Squared Error (RMSE) on Training set --->',round(np.sqrt(mean_squared_error(Train_Y, pred1)),20))

    print('\n{}Testing Set Metrics{}'.format('-'*20, '-'*20))
    print('\nR2-Score on Testing set --->',round(r2_score(Test_Y, pred2),20))
    print('Residual Sum of Squares (RSS) on Testing set  --->',round(np.sum(np.square(Test_Y-pred2)),20))
    print('Mean Squared Error (MSE) on Testing set       --->',round(mean_squared_error(Test_Y, pred2),20))
    print('Root Mean Squared Error (RMSE) on Testing set --->',round(np.sqrt(mean_squared_error(Test_Y, pred2)),20))
    print('\n{}Residual Plots{}'.format('-'*20, '-'*20))

    Model_Evaluation_Comparison_Matrix.loc[n,'Train-R2']  = round(r2_score(Train_Y, pred1),20)
    Model_Evaluation_Comparison_Matrix.loc[n,'Test-R2']   = round(r2_score(Test_Y, pred2),20)
    Model_Evaluation_Comparison_Matrix.loc[n,'Train-RSS'] = round(np.sum(np.square(Train_Y-pred1)),20)
    Model_Evaluation_Comparison_Matrix.loc[n,'Test-RSS']  = round(np.sum(np.square(Test_Y-pred2)),20)
    Model_Evaluation_Comparison_Matrix.loc[n,'Train-MSE'] = round(mean_squared_error(Train_Y, pred1),20)
    Model_Evaluation_Comparison_Matrix.loc[n,'Test-MSE']  = round(mean_squared_error(Test_Y, pred2),20)
    Model_Evaluation_Comparison_Matrix.loc[n,'Train-RMSE']= round(np.sqrt(mean_squared_error(Train_Y, pred1)),20)
    Model_Evaluation_Comparison_Matrix.loc[n,'Test-RMSE'] = round(np.sqrt(mean_squared_error(Test_Y, pred2)),20)

    plt.figure(figsize=[15,4])

    plt.subplot(1,2,1)
    sns.distplot((Train_Y - pred1))
    plt.title('Error Terms')
    plt.xlabel('Errors')

    plt.subplot(1,2,2)
    plt.scatter(Train_Y,pred1)
    plt.plot([Train_Y.min(),Train_Y.max()],[Train_Y.min(),Train_Y.max()], 'r--')
    plt.title('Actual vs Predicted (Train)')
    plt.xlabel('Actual')
    plt.ylabel('Predicted')
    plt.show()


### Modeling

### Multiple Linear Regression (MLR)

We train a basic multiple linear regression model using all features and evaluate its performance on train and test sets.

In [ ]:
# Multiple Linear Regression(MLR)
#Linear Regression

MLR = LinearRegression().fit(Train_X_std,Train_Y)
pred1 = MLR.predict(Train_X_std)
pred2 = MLR.predict(Test_X_std)

print('{}{}\033[1m Evaluating Multiple Linear Regression Model \033[0m{}{}\n'.format('<'*3,'-'*35 ,'-'*35,'>'*3))
print('The Coeffecient of the Regresion Model was found to be ',MLR.coef_)
print('The Intercept of the Regresion Model was found to be ',MLR.intercept_)

Evaluate(0, pred1, pred2)

In [ ]:
# Ridge Regression Model
#Creating a Ridge Regression model

RLR = Ridge().fit(Train_X_std,Train_Y)
pred1 = RLR.predict(Train_X_std)
pred2 = RLR.predict(Test_X_std)

print('{}{}\033[1m Evaluating Ridge Regression Model \033[0m{}{}\n'.format('<'*3,'-'*35 ,'-'*35,'>'*3))
print('The Coeffecient of the Regresion Model was found to be ',MLR.coef_)
print('The Intercept of the Regresion Model was found to be ',MLR.intercept_)

Evaluate(1, pred1, pred2)

In [ ]:
# Lasso Regression Model
#Creating a Ridge Regression model

LLR = Lasso().fit(Train_X_std,Train_Y)
pred1 = LLR.predict(Train_X_std)
pred2 = LLR.predict(Test_X_std)

print('{}{}\033[1m Evaluating Lasso Regression Model \033[0m{}{}\n'.format('<'*3,'-'*35 ,'-'*35,'>'*3))
print('The Coeffecient of the Regresion Model was found to be ',MLR.coef_)
print('The Intercept of the Regresion Model was found to be ',MLR.intercept_)

Evaluate(2, pred1, pred2)

In [ ]:
# Elastic-Net Regression
#Creating a ElasticNet Regression model

ENR = ElasticNet().fit(Train_X_std,Train_Y)
pred1 = ENR.predict(Train_X_std)
pred2 = ENR.predict(Test_X_std)

print('{}{}\033[1m Evaluating Elastic-Net Regression Model \033[0m{}{}\n'.format('<'*3,'-'*35 ,'-'*35,'>'*3))
print('The Coeffecient of the Regresion Model was found to be ',MLR.coef_)
print('The Intercept of the Regresion Model was found to be ',MLR.intercept_)

Evaluate(3, pred1, pred2)

In [ ]:
Trr = []
Tss = []
n_degree = 7

for i in range(2, n_degree):
    poly_reg = PolynomialFeatures(degree=i)
    X_poly = poly_reg.fit_transform(Train_X_std)
    X_poly1 = poly_reg.fit_transform(Test_X_std)
    LR = LinearRegression()
    LR.fit(X_poly, Train_Y)

    pred1 = LR.predict(X_poly)
    Trr.append(np.sqrt(mean_squared_error(Train_Y, pred1)))

    pred2 = LR.predict(X_poly1)
    Tss.append(np.sqrt(mean_squared_error(Test_Y, pred2)))

plt.figure(figsize=[15, 6])
plt.subplot(1, 2, 1)
plt.plot(range(2, n_degree), Trr, label='Training')
plt.plot(range(2, n_degree), Tss, label='Testing')
plt.title('Polynomial Regression Fit')
plt.xlabel('Degree')
plt.ylabel('RMSE')
plt.grid()
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(range(2, n_degree), Trr, label='Training')
plt.plot(range(2, n_degree), Tss, label='Testing')
plt.title('Polynomial Regression Fit')
plt.ylim([0, 1e15])
plt.xlabel('Degree')
plt.ylabel('RMSE')
plt.grid()
plt.legend()
plt.show()

In [ ]:
poly_reg = PolynomialFeatures(degree=5)
X_poly = poly_reg.fit_transform(Train_X_std)
X_poly1 = poly_reg.fit_transform(Test_X_std)
PR = LinearRegression()
PR.fit(X_poly, Train_Y)

pred1 = PR.predict(X_poly)
pred2 = PR.predict(X_poly1)

print('{}{}\033[1m Evaluating Polynomial Regression Model \033[0m{}{}\n'.format('<' * 3, '-' * 35, '-' * 35, '>' * 3))
print('The Coeffecient of the Regresion Model was found to be ', PR.coef_)
print('The Intercept of the Regresion Model was found to be ', PR.intercept_)

Evaluate(4, pred1, pred2)

### Comparing Model Evaluation Metrics

We compare the performance metrics (R2, RMSE, etc.) of all trained models to identify the best performing one.

In [ ]:
EMC = Model_Evaluation_Comparison_Matrix.copy()
EMC.index = ['Multiple Linear Regression (MLR)', 'Ridge Linear Regression (RLR)', 'Lasso Linear Regression (LLR)', 'Elastic-Net Regression (ENR)', 'Polynomial Regression (PNR)']
EMC

In [ ]:
R2 = round(EMC['Train-R2'].sort_values(ascending=True), 4)
plt.hlines(y=R2.index, xmin=0, xmax=R2.values)
plt.plot(R2.values, R2.index, 'o')
plt.title('R2-Scores Comparison for various Regression Models')
plt.xlabel('R2-Score')
for i, v in enumerate(R2):
    plt.text(v + 0.02, i - 0.05, str(v * 100), color='blue')
plt.xlim([0, 1.1])
plt.show()

In [ ]:
# Root Mean SquaredError Comparison for different Regression Models

cc = Model_Evaluation_Comparison_Matrix.columns.values
s=5
plt.bar(np.arange(5), Model_Evaluation_Comparison_Matrix[cc[6]].values, width=0.3, label='RMSE (Training)')
plt.bar(np.arange(5)+0.3, Model_Evaluation_Comparison_Matrix[cc[7]].values, width=0.3, label='RMSE (Testing)')
plt.xticks(np.arange(5),EMC.index, rotation =35)
plt.legend()
plt.ylim([0,1.25e6])
plt.show()